# Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings as w
import seaborn as sns
import matplotlib.pyplot as plt
w.filterwarnings('ignore')

# Import File

In [2]:
df=pd.read_csv('../Data/raw_salary.csv')

In [3]:
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


# Preprocessing and Feature Engineering
![ppe](https://media.geeksforgeeks.org/wp-content/uploads/20250127140210993694/data_preprocessing-660.webp)

### Pre-processing and Feature Engineering

This is the pre-processing and feature engineering stage.  
Here, I will be creating new features and fine-tuning the existing features based on the model.  
I will apply different approaches.  
As we move further into this segment, I will describe each approach.


In [6]:
missing_col = ['workclass', 'occupation', 'native-country']
for col in missing_col:
    df[col] = df[col].replace('?', 'unknown')

In [19]:
Q1=df['overall_multiplier'].quantile(0.25)
Q3=df['overall_multiplier'].quantile(0.75)
IQR=Q3-Q1
lower_bound= Q1 - 1.5 * IQR
upper_bound= Q3 + 1.5 * IQR
df=df[(df['overall_multiplier'] >= lower_bound) & (df['overall_multiplier'] <= upper_bound)]

### 5>Correlation

# MODEL

## Baseline Model
<img src="https://miro.medium.com/v2/resize:fit:1400/1*fHlHur4KBdGoOkYY1WLRqw.png" width="700">


A baseline model is a simple model used as a starting point in a machine learning project.<br>
It helps you set a benchmark so you can compare how well more complex models perform.

In [17]:
# importing libraries
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.pipeline import make_pipeline, Pipeline
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


In [19]:
# creating X and y an instance of df
X=df.drop(columns='income',axis=1) # has all the columns other than salary and the columns highly related to salary
y=df['income'] # has only one column i.e salary

# creating variable [num and cat] of X based on the data types, we will further you these variable for scaling and encoding.
num=X.select_dtypes(include=['number']).columns.tolist()
cat=X.select_dtypes(include=['object']).columns.tolist()

# creating train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42) 

# Making a ColumnTransformer and using StandardScaler and OneHotEncoder to Standardize and Encode the features.
# We never perform any Transformation on target remember this.
preprocessing=make_column_transformer((StandardScaler(),num),
                                     (OneHotEncoder(handle_unknown='ignore'),cat),remainder='drop')

# Creating a pipeline and feeding ColumnTransformer followed by Linear Regression
model= make_pipeline ( preprocessing, LogisticRegression () )

# fitting in model
model.fit(X_train,y_train)

# making prediction
y_pred=model.predict(X_test)

#Evaluation 

# To check if model is underfitting or overfitting
print(f'TRAINING SCORE: {model.score(X_train,y_train)}')
print(f'TEST SCORE: {model.score(X_test,y_test)}')
# general evaluation metices
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')  # 'weighted' handles class imbalance
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion Matrix:")
print(cm)


TRAINING SCORE: 0.8528139636065826
TEST SCORE: 0.8569966219674481
Accuracy : 0.8570
Precision: 0.8509
Recall   : 0.8570
F1-score : 0.8520
Confusion Matrix:
[[6981  498]
 [ 899 1391]]


In [ ]:
# creating X and y an instance of df
X=df.drop(columns='income',axis=1) # has all the columns other than salary and the columns highly related to salary
y=df['income'] # has only one column i.e salary

# creating variable [num and cat] of X based on the data types, we will further you these variable for scaling and encoding.
num=X.select_dtypes(include=['number']).columns.tolist()
cat=X.select_dtypes(include=['object']).columns.tolist()

# creating train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42) 

# Making a ColumnTransformer and using StandardScaler and OneHotEncoder to Standardize and Encode the features.
# We never perform any Transformation on target remember this.
num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='mean')),
                         ('scaler', StandardScaler())])
cat_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                         ('scaler', OneHotEncoder(handle_unknown='ignore'))])
preprocessing= ColumnTransformer([('mun_pipe', num_pipeline, num),
                                  ('cat_pipe', cat_pipeline, cat)])
# Creating a pipeline and feeding ColumnTransformer followed by Linear Regression
lr= make_pipeline ( preprocessing, LogisticRegression () )
dtc= make_pipeline ( preprocessing, DecisionTreeClassifier () )
lgbm= make_pipeline ( preprocessing, lgb.LGBMClassifier () )

model = VotingClassifier([('lr',lr),
                          ('dtc', dtc),
                          ('lgbm', lgbm)])

model.fit(X_train,y_train)

# making prediction
y_pred=model.predict(X_test)

print(f'TRAINING SCORE: {model.score(X_train,y_train)}')
print(f'TEST SCORE: {model.score(X_test,y_test)}')
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion Matrix:")
print(cm)


[LightGBM] [Info] Number of positive: 9397, number of negative: 29676
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008326 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 802
[LightGBM] [Info] Number of data points in the train set: 39073, number of used features: 98
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.240499 -> initscore=-1.149948
[LightGBM] [Info] Start training from score -1.149948
TRAINING SCORE: 0.8966293860210376
TEST SCORE: 0.8730678677449074
Accuracy : 0.8731
Precision: 0.8684
Recall   : 0.8731
F1-score : 0.8690
Confusion Matrix:
[[7046  433]
 [ 807 1483]]
